**LogisticRegression и LinearRegression**


Импорт библиотек

In [141]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_percentage_error as MAPE

Загрузка данных пресс-релизов

In [3]:
key = pd.read_csv('https://raw.githubusercontent.com/HSEAi2025Team-7/keyRateForecast/dev/cbr.csv')

In [4]:
key.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   date             50 non-null     object 
 1   change_key_rate  50 non-null     object 
 2   basis_points     29 non-null     float64
 3   key_rate         48 non-null     float64
 4   inflation        38 non-null     float64
dtypes: float64(3), object(2)
memory usage: 2.1+ KB


Заполнение пропусков

In [5]:
# заполняем пропуски вручную
idx = key[key["inflation"].isna()].index
new_vals = [11, 14.3, 15.9, 17.8, 17.8, 9.5, 6, 5.7, 5.2, 4, 3.1, 3.1]
for i, val in zip(idx, new_vals):
    key.at[i, "inflation"] = val
# заполняем пропуски вручную
idx = key[key["key_rate"].isna()].index
new_vals_k = [20, 20]
for i, val_k in zip(idx, new_vals_k):
    key.at[i, "key_rate"] = val_k
# заполняем медианой basis_point
key['basis_points'] = key['basis_points'].fillna('0')

Добавление числовых признаков

In [6]:
# добавляем колонку цель по инфляции = 4
key['inflation_target'] = 4
# добавляем колонку разница между инфляцией и целью по инфляции
key['difference'] = key['inflation'] - key['inflation_target']
# добавляем колонку разница межды инфляцией
key['inflation_diff'] = key['inflation'].diff().shift(-1)
# заполняем пропуск медианой
med_inflation_diff = key["inflation_diff"].median()
key["inflation_diff"] = key["inflation_diff"].fillna(med_inflation_diff)
# прошлая ключевая ставка
key['key_rate_last'] = key['key_rate'].shift(-1)
med_key_rate_last = key['key_rate_last'].median()
key['key_rate_last'] = key['key_rate_last'].fillna(med_key_rate_last)
# на сколько изменилась ставка
key['key_rate_last_diff'] = key['key_rate_last'] - key['key_rate_last'].shift(-1)
med_key_rate_diff = key["key_rate_last_diff"].median()
key["key_rate_last_diff"] = key["key_rate_last_diff"].fillna(med_key_rate_diff)

так как в пресс релизах в качестве признаков в основном указана назначенная ключевая ставка, на сколько базовых пунктов изменилась и годовая инфляция, добавили такие признаки как:

inflation_target цель по инфляции = 4%

difference разница между инфляцией и целью по инфляции

inflation_diff разница межды инфляцией прошлой и текущей

key_rate_last прошлая ключевая ставка

key_rate_last_diff на сколько изменилась прошлая ставка

In [ ]:
key.isna().sum()

,0
date,0
change_key_rate,0
basis_points,0
key_rate,0
inflation,0
inflation_target,0
difference,0
inflation_diff,0
key_rate_last,0
key_rate_last_diff,0


In [ ]:
key.head(50)

,date,change_key_rate,basis_points,key_rate,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff
0,24 октября 2025,снизить,50.0,16.50,8.2,4,4.2,0.0,17.00,-1.00
1,12 сентября 2025,снизить,100.0,17.00,8.2,4,4.2,1.0,18.00,-2.00
2,25 июля 2025,снизить,200.0,18.00,9.2,4,5.2,0.6,20.00,-1.00
3,6 июня 2025,снизить,100.0,20.00,9.8,4,5.8,0.5,21.00,0.00
4,25 апреля 2025,сохранить,0,21.00,10.3,4,6.3,-0.1,21.00,0.00
5,21 марта 2025,сохранить,0,21.00,10.2,4,6.2,-0.2,21.00,0.00
6,14 февраля 2025,сохранить,0,21.00,10.0,4,6.0,-0.5,21.00,0.00
7,20 декабря 2024,сохранить,0,21.00,9.5,4,5.5,-1.1,21.00,2.00
8,25 октября 2024,повысить,200.0,21.00,8.4,4,4.4,0.6,19.00,1.00
9,13 сентября 2024,повысить,100.0,19.00,9.0,4,5.0,-2.0,18.00,2.00


Разбиение на матрицу признаков и целевую переменную

In [7]:
# делим на матрицу признаков и целевую переменную для логистической регрессии
X1 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y1 = key['change_key_rate']

In [8]:
# делим на матрицу признаков и целевую переменную для линейной регрессии
X2 = key[['inflation', 'inflation_target', 'difference', 'inflation_diff', 'key_rate_last', 'key_rate_last_diff']]
y2 = key['key_rate']

In [ ]:
X1.head()

,inflation,inflation_target,difference,inflation_diff,key_rate_last,key_rate_last_diff
0,8.2,4,4.2,0.0,17.0,1.0
1,8.2,4,4.2,1.0,18.0,2.0
2,9.2,4,5.2,0.6,20.0,1.0
3,9.8,4,5.8,0.5,21.0,0.0
4,10.3,4,6.3,-0.1,21.0,0.0


In [ ]:
y1.head()

,change_key_rate
0,снизить
1,снизить
2,снизить
3,снизить
4,сохранить


Обучаем логистическую модель

In [146]:
# Разделение данных
X1_train, X1_test, y1_train, y1_test = train_test_split(
    X1, y1, test_size=0.20, random_state=42 , stratify=y1)

# Масштабирование
scaler = StandardScaler()
X1_train_scaled = scaler.fit_transform(X1_train)
X1_test_scaled = scaler.transform(X1_test)

# Создание модели с фиксированными параметрами
model = LogisticRegression()

# Обучение модели
model.fit(X1_train_scaled, y1_train)

# Оценка модели на тесте

y1_pred = model.predict(X1_test_scaled)

 Использовали stratify=y1 для разбивки т.к. нам важно сохранять распределение классов целевой переменной в тренировочной и тестовой выборках.

In [147]:
print("Accuracy:", accuracy_score(y1_test, y1_pred))
print("f1:", f1_score(y1_test, y1_pred, average='micro'))

Accuracy: 0.7
f1: 0.7


Общая точность Accuracy = 0.7 показывает, что 70% всех предсказаний модель делает правильно.

F1 метрика = 0.7 — среднее между точностью положительных предсказаний и долей найденных истинных положительных.

То, что Accuracy и F1 примерно совпадают, говорить о том, что модель различает классы при предсказании.

Вывод результата предсказаний для наглядности.

In [148]:
print(y1_pred)
print(y1_test)

['снизить' 'повысить' 'сохранить' 'снизить' 'повысить' 'повысить'
 'сохранить' 'повысить' 'сохранить' 'снизить']
26      снизить
18     повысить
21    сохранить
29      снизить
37     повысить
43    сохранить
23    сохранить
42    сохранить
15     повысить
1       снизить
Name: change_key_rate, dtype: object


Обучение модели линейной регресси для предсказания значения ключевой ставки.

In [137]:
# Разделение данных
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.20, random_state=42
    )

    # Масштабирование
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

    # Создание модели с фиксированными параметрами
model = LinearRegression()

                # Обучение модели
model.fit(X2_train_scaled, y2_train)

                # Оценка модели на тесте

y2_pred = model.predict(X2_test_scaled)

In [144]:
mape = MAPE(y2_test, y2_pred)*100
# Вывод результатов
print(f"MAPE: {mape:.1f}%")

MAPE: 15.6%


в среднем предсказанные значения отклоняются от истинных на ~15.6%. То есть,ошибка около 15–16% от реального.

In [145]:
print(y2_pred)
print(y2_test)

[15.89843027  6.09725326 14.14038464  4.5664096  15.63980539  2.91335774
  7.01666638  7.51394648 31.05366623  7.89035252]
13    16.00
39     5.50
30    14.00
45     4.25
17    13.00
48     5.50
26     7.50
25     7.50
32    20.00
19     8.50
Name: key_rate, dtype: float64


## Модели KNN и SVM

In [ ]:
import pandas as pd

# считываем датасеты с обработанным текстом и числовыми показателями
data = pd.read_csv('../EDA/key_inf_dollar.csv', parse_dates=['date'])
text = pd.read_json('../EDA/cbr_key-rate_press_releases_processed.json')

In [ ]:
text.dropna(subset=['decision'], inplace=True)
text.reset_index(drop=True, inplace=True)

Из текстовых данных необходимо взять: количество слов, решение по ставке и дату, чтобы сопоставить с числовыми данными

In [ ]:
import dateparser

# переводим дату в формат дат
text["date"] = text["date"].apply(lambda x: dateparser.parse(x))

In [ ]:
#добавим к имеющемуся датасету data - количество слов в каждом пресс-релизе, решение по ставке

# сократим даты до месяца для data
data['year_month'] = data['date'].dt.to_period('M')

# сокртим даты до месяца для text
text['year_month'] = text['date'].dt.to_period('M')

# добавим количество слов в пресс-релизе в отдельную колонку
text['num_tokens'] = text['text_tokens'].apply(len)

#добавляем столбец со ставкой прошлого месяца
data['key_rate_prev'] = data['key-rate'].shift(-1)

#добавим к data столбец с решением, количеством слов
text_small = text[['year_month', 'decision', 'num_tokens']]
data = data.merge(text_small, on='year_month', how='left', suffixes=('', '_from_text'))

# убираем лишний столбец с индексами
data = data.drop(['Unnamed: 0'], axis=1)

# убираем строки, в которых нет решения
data = data.dropna(subset=['decision'])

In [ ]:
data.reset_index(drop=True, inplace=True)

In [ ]:
data.head(10)

,date,key-rate,inflation,dollar_rate,year_month,key_rate_prev,decision,num_tokens
0,2025-09-01,17.0,7.98,80.4261,2025-09,18.0,-1.0,485.0
1,2025-07-01,18.0,8.79,78.5284,2025-07,20.0,-1.0,508.0
2,2025-06-01,20.0,9.40,79.1285,2025-06,21.0,-1.0,467.0
3,2025-04-01,21.0,10.23,85.4963,2025-04,21.0,0.0,486.0
4,2025-03-01,21.0,10.34,88.2568,2025-03,21.0,1.0,539.0
5,2025-02-01,21.0,10.06,97.8107,2025-02,21.0,1.0,536.0
6,2024-12-01,21.0,9.52,107.1758,2024-12,21.0,1.0,542.0
7,2024-10-01,21.0,8.54,93.2221,2024-10,19.0,1.0,475.0
8,2024-09-01,19.0,8.63,90.0013,2024-09,18.0,1.0,523.0
9,2024-07-01,18.0,9.13,87.2972,2024-07,16.0,1.0,458.0


In [ ]:
X = data[['inflation', 'dollar_rate', 'num_tokens', 'key_rate_prev']]
y = data['decision']

In [ ]:
# разделяем датасет на трейн и тест

from sklearn.model_selection import train_test_split
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
from sklearn.preprocessing import StandardScaler

# делаем объект масштабирования
scaler = StandardScaler()
# считаем параметры стандартизации на трейне
model = scaler.fit(Xtrain)
# стандартизируем признаки для тест и трейн
scaler_data_train = scaler.transform(Xtrain)
scaler_data_test = scaler.transform(Xtest)
# создаем обратно датафрейм со стандартизированными признаками
Xtest_scaled = pd.DataFrame(data=scaler_data_test, columns=Xtest.columns, index=Xtest.index)
Xtrain_scaled = pd.DataFrame(data=scaler_data_train, columns=Xtrain.columns, index=Xtrain.index)

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score

model = SVC(kernel = 'linear')

model.fit(scaler_data_train, ytrain)

pred = model.predict(scaler_data_test)

print("Accuracy:", accuracy_score(ytest, pred))
print("F1 weighted:", f1_score(ytest, pred, average='weighted'))

Accuracy: 0.5769230769230769
F1 weighted: 0.5507692307692308


Модель дала средние результаты, поэтому следует дальше попробовать изучить другие модели.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import f1_score, accuracy_score

# модель KNN
knn = KNeighborsClassifier()

# параметры для подбора
param_grid = {
    'n_neighbors': range(1, 21),  # количество соседей
    'weights': ['uniform', 'distance'],  # равные веса или взвешенные по расстоянию
    'metric': ['euclidean', 'manhattan']  # тип расстояния
}

# GridSearch с 5-кратной кросс-валидацией и F1 (weighted) как метрика
grid_search = GridSearchCV(knn, param_grid, cv=5, scoring='f1_weighted')

# обучаем модель
grid_search.fit(scaler_data_train, ytrain)

# выводим лучшие параметры
print("Лучшие параметры:", grid_search.best_params_)

# выбираем лучшую модель
best_knn = grid_search.best_estimator_

# делаем предсказание на тесте
y_pred = best_knn.predict(scaler_data_test)

# выводим метрики
print("Accuracy:", accuracy_score(ytest, y_pred))
print("F1 weighted:", f1_score(ytest, y_pred, average='weighted'))


Лучшие параметры: {'metric': 'manhattan', 'n_neighbors': 11, 'weights': 'distance'}
Accuracy: 0.8076923076923077
F1 weighted: 0.8095652173913043


In [ ]:
y_pred

array([-1., -1., -1., -1.,  1.,  1.,  0., -1.,  1.,  1.,  1.,  1.,  1.,
       -1., -1.,  1., -1.,  1.,  1., -1.,  0., -1.,  1., -1., -1.,  1.])